# 🐍 Day 25 · Python Basics — async / await & JSON Messages (MCP Fundamentals)

> **Goal of this notebook:** MCP clients and servers talk with JSON messages and use `async` code so they can wait for many things at once. Learn both from zero.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. Normal (sync) code vs `async` code
2. `async def` and `await`
3. Running many tasks together with `asyncio.gather`
4. JSON-RPC messages — the language of MCP
5. 🔗 A tiny MCP-style server that handles requests

## 1. Why `async`?
Normal code waits for one thing at a time, like a single cashier. **Async** code is like a waiter: while one dish cooks, they take the next order.
It is perfect for programs that spend time **waiting** — for the internet, a server or an LLM.

> 💡 In Jupyter you can write `await` directly in a cell. In a `.py` file you start async code with `asyncio.run(main())`.

In [ ]:
import time

def make_tea():
    time.sleep(1)          # waiting... nothing else can happen
    return "tea ready"

start = time.time()
print(make_tea(), make_tea())
print(f"Normal code took {time.time() - start:.1f} s")

## 2. `async def` and `await`
- `async def` creates a **coroutine** (a function that can pause).
- `await` means *pause here until this finishes, and let other work run meanwhile*.

In [ ]:
import asyncio

async def make_coffee(name):
    print(f"☕ start {name}")
    await asyncio.sleep(1)        # async waiting: others may run now
    print(f"✅ done {name}")
    return f"{name} ready"

result = await make_coffee("latte")      # top-level await works in Jupyter
print(result)

## 3. Many tasks at once with `asyncio.gather`

In [ ]:
start = time.time()
results = await asyncio.gather(
    make_coffee("latte"),
    make_coffee("mocha"),
    make_coffee("espresso"),
)
print(results)
print(f"3 coffees in {time.time() - start:.1f} s instead of 3 s 🚀")

## 4. JSON-RPC messages
MCP uses **JSON-RPC 2.0**. A request names a `method` and gives `params`; the response has the same `id` and a `result`.
```json
{"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "add", "arguments": {"a": 2, "b": 3}}}
```

In [ ]:
import json

request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {"name": "add", "arguments": {"a": 2, "b": 3}},
}
text = json.dumps(request)            # what actually travels over the wire (text)
print(text)

back = json.loads(text)               # the other side turns it back into a dict
print(back["params"]["arguments"])

## 5. 🔗 A tiny MCP-style server
A real one is built with `FastMCP` and `@mcp.tool`. This one shows what happens inside: list the tools, or call one.

In [ ]:
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

def list_files(folder: str) -> list:
    "List files in a folder."
    import os
    return sorted(os.listdir(folder))[:5]

TOOLS = {"add": add, "list_files": list_files}

async def handle(message_text):
    msg = json.loads(message_text)
    if msg["method"] == "tools/list":
        result = [{"name": n, "description": f.__doc__} for n, f in TOOLS.items()]
    elif msg["method"] == "tools/call":
        tool = TOOLS[msg["params"]["name"]]
        result = tool(**msg["params"]["arguments"])
    else:
        result = f"unknown method {msg['method']}"
    await asyncio.sleep(0.1)                         # pretend network delay
    return json.dumps({"jsonrpc": "2.0", "id": msg["id"], "result": result})

print(await handle(json.dumps({"jsonrpc": "2.0", "id": 1, "method": "tools/list"})))
print(await handle(json.dumps(request)))
print(await handle(json.dumps({"jsonrpc": "2.0", "id": 3, "method": "tools/call",
                               "params": {"name": "list_files", "arguments": {"folder": "."}}})))

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Write `async def wait_and_say(text, seconds)` that sleeps then returns the text. Run two of them with `gather`.
2. Build a JSON-RPC request that calls `add` with `a=10, b=20` and send it to `handle`.
3. Add a new tool `multiply` to `TOOLS` and call it.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
async def wait_and_say(text, seconds):
    await asyncio.sleep(seconds)
    return text
print(await asyncio.gather(wait_and_say("hi", 1), wait_and_say("there", 1)))

req = {"jsonrpc": "2.0", "id": 9, "method": "tools/call", "params": {"name": "add", "arguments": {"a": 10, "b": 20}}}
print(await handle(json.dumps(req)))

def multiply(a: int, b: int) -> int:
    "Multiply two numbers."
    return a * b
TOOLS["multiply"] = multiply
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Async function | `async def fetch(): ...` |
| Wait without blocking | `await asyncio.sleep(1)` |
| Run together | `await asyncio.gather(a(), b())` |
| Run in a .py file | `asyncio.run(main())` |
| Dict -> JSON text | `json.dumps(msg)` |
| JSON-RPC request | `{'jsonrpc': '2.0', 'id': 1, 'method': 'tools/call'}` |

## ➡️ What's next?
You now know the Python used in **Day_25_MCP_Fundamentals_and_Servers**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀